# XAI-ED — Notebook 3: XAI Deep Dive

This notebook provides an in-depth analysis of all three explainability layers:

1. **SHAP** — Global beeswarm plots, local waterfall charts, feature stability across CV folds
2. **LIME** — Global importance, local explanations, side-by-side comparison with SHAP
3. **Counterfactuals** — Flip rate by risk tier, actionability analysis, path visualisation
4. **SHAP vs LIME Agreement** — Cross-method stability metric
5. **Fairness Analysis** — Group metrics across all three demographic axes

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split

from src.config import FEATURE_COLUMNS, TARGET_COLUMN, DEMOGRAPHIC_COLUMNS
from src.data_loader import load_dataset
from src.train_model import train
from src.explain import explain_with_shap, _extract_class1_shap
from src.lime_explain import explain_with_lime
from src.counterfactual import generate_counterfactual
from src.fairness import compute_fairness_metrics
from src.translator import _risk_tier

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

FRIENDLY = {
    'study_time_min': 'Study Time',
    'practice_completion_rate': 'Practice Completion',
    'avg_quiz_score': 'Avg Quiz Score',
    'quiz_attempts': 'Quiz Attempts',
    'hint_usage_rate': 'Hint Usage',
    'attendance_rate': 'Attendance Rate',
    'days_since_last_activity': 'Days Inactive',
    'stress_index': 'Stress Index',
    'prereq_mastery': 'Prereq Mastery',
    'device_reliability': 'Device Reliability',
}
print('Libraries loaded.')

In [ ]:
import pandas as pd
df_full = pd.read_csv('../data/student_data.csv')
X, y    = load_dataset('../data/student_data.csv')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

_, test_idx = train_test_split(
    range(len(df_full)), test_size=0.2, random_state=42,
    stratify=df_full[TARGET_COLUMN]
)
demo_test = df_full.iloc[test_idx].reset_index(drop=True)[DEMOGRAPHIC_COLUMNS]

print('Training RF model for XAI analysis...')
trained_rf = train('rf', X_train, y_train)
rf_pipeline = trained_rf.pipeline
print('RF model ready.')

## 1. SHAP Global Analysis

In [ ]:
preprocess = rf_pipeline.named_steps['preprocess']
clf        = rf_pipeline.named_steps['clf']

X_ex_raw  = X_test.sample(n=150, random_state=42)
Xe        = preprocess.transform(X_ex_raw)

explainer  = shap.TreeExplainer(clf)
shap_raw   = explainer.shap_values(Xe)
sv         = _extract_class1_shap(shap_raw)

print(f'SHAP values shape: {sv.shape}')

# Beeswarm summary
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, Xe, feature_names=[FRIENDLY.get(f, f) for f in FEATURE_COLUMNS],
                  show=False, plot_type='dot')
plt.title('SHAP Beeswarm — Random Forest (Global)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/shap_beeswarm_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP mean absolute importance (bar chart)
mean_abs = np.abs(sv).mean(axis=0)
order    = np.argsort(mean_abs)[::-1]
feat_labels = [FRIENDLY.get(FEATURE_COLUMNS[i], FEATURE_COLUMNS[i]) for i in order]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(feat_labels[::-1], mean_abs[order[::-1]], color='#42a5f5', edgecolor='white')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Global Feature Importance (Random Forest)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features by |SHAP|:')
for i in order[:5]:
    print(f'  {FEATURE_COLUMNS[i]:<35} mean |SHAP| = {mean_abs[i]:.4f}')

## 2. Local SHAP — Example Student Waterfall

In [ ]:
# Pick an at-risk student for demonstration
proba_test = rf_pipeline.predict_proba(X_test)[:, 1]
at_risk    = np.where(proba_test < 0.4)[0]
student_i  = at_risk[0] if len(at_risk) > 0 else 0

student_row  = X_test.iloc[student_i]
student_prob = proba_test[student_i]
Xe_student   = preprocess.transform(pd.DataFrame([student_row]))
sv_student   = _extract_class1_shap(explainer.shap_values(Xe_student))[0]

print(f'Student #{student_i}: mastery prob = {student_prob:.3f} ({_risk_tier(student_prob)})')

# Waterfall-style bar chart
order    = np.argsort(np.abs(sv_student))[::-1]
top_idx  = order[:8]
vals     = sv_student[top_idx]
labels   = [FRIENDLY.get(FEATURE_COLUMNS[i], FEATURE_COLUMNS[i]) for i in top_idx]
colors   = ['#42a5f5' if v > 0 else '#ef5350' for v in vals]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(labels[::-1], vals[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0, color='white', lw=0.8)
ax.set_xlabel('SHAP Value (impact on mastery probability)')
ax.set_title(f'Local SHAP — Student #{student_i} (p={student_prob:.3f}, {_risk_tier(student_prob)})',
             fontweight='bold')
for bar, val in zip(ax.patches, vals[::-1]):
    ax.text(val + 0.001 if val >= 0 else val - 0.001,
            bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/shap_local_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. LIME vs SHAP Agreement Analysis

In [ ]:
import os
# Load pre-computed CSVs if available; otherwise skip
shap_csv_path = '../outputs/explanations/rf_local_explanations.csv'
lime_csv_path = '../outputs/explanations/rf_lime_local_explanations.csv'

if os.path.exists(shap_csv_path) and os.path.exists(lime_csv_path):
    shap_df = pd.read_csv(shap_csv_path)
    lime_df = pd.read_csv(lime_csv_path)
    n = min(len(shap_df), len(lime_df))

    agree_counts = []
    for i in range(n):
        s_feats = set(shap_df.iloc[i]['top_features'].split('; ')[:5])
        l_feats = set(lime_df.iloc[i]['top_features'].split('; ')[:5])
        agree_counts.append(len(s_feats & l_feats))

    agree_arr = np.array(agree_counts)
    print(f'Comparing {n} students')
    print(f'Mean SHAP-LIME top-5 agreement: {agree_arr.mean():.2f} / 5 features')
    print(f'Full agreement (5/5): {(agree_arr == 5).mean():.1%}')
    print(f'Low agreement (≤2/5): {(agree_arr <= 2).mean():.1%}')

    fig, ax = plt.subplots(figsize=(7, 4))
    unique, counts = np.unique(agree_counts, return_counts=True)
    ax.bar(unique, counts / n * 100, color='#42a5f5', edgecolor='white')
    ax.set_xlabel('# Features in common (top 5)')
    ax.set_ylabel('% of Students')
    ax.set_xticks([0, 1, 2, 3, 4, 5])
    ax.set_title('SHAP vs LIME Top-5 Feature Agreement Distribution', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/shap_lime_agreement.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Pre-computed CSVs not found. Run scripts/run_all.py first.')
    print(f'  Looking for: {shap_csv_path}')
    print(f'  Looking for: {lime_csv_path}')

## 4. Counterfactual Analysis

In [ ]:
# Compute counterfactuals for a sample of at-risk students
at_risk_idx = np.where(rf_pipeline.predict_proba(X_test)[:, 1] < 0.5)[0]
sample      = at_risk_idx[:min(80, len(at_risk_idx))]

cf_results = []
for idx in sample:
    row    = X_test.iloc[int(idx)]
    prob   = float(rf_pipeline.predict_proba(pd.DataFrame([row]))[:, 1][0])
    cf     = generate_counterfactual(rf_pipeline, row.copy())
    cf_results.append({
        'tier':      _risk_tier(prob),
        'status':    cf['status'],
        'base_prob': cf['base_prob'],
        'new_prob':  cf['new_prob'],
        'delta':     cf['delta_prob'],
        'steps':     cf['steps_taken'],
        'n_edits':   len(cf['edits']),
    })

cf_df = pd.DataFrame(cf_results)
print(f'Analysed {len(cf_df)} at-risk students')
print(f'Flip rate: {(cf_df["status"] == "flipped").mean():.1%}')
print(f'\nStatus breakdown:')
print(cf_df['status'].value_counts())
print(f'\nMean steps taken (flipped cases): {cf_df[cf_df["status"]=="flipped"]["steps"].mean():.1f}')

In [ ]:
# Flip rate by risk tier
tier_order = ['High Risk', 'At Risk', 'Borderline']
flip_by_tier = []
for tier in tier_order:
    sub = cf_df[cf_df['tier'] == tier]
    if len(sub) > 0:
        flip_by_tier.append({
            'Tier': tier,
            'N': len(sub),
            'Flip Rate': (sub['status'] == 'flipped').mean(),
            'Mean Δ Prob': sub['delta'].mean(),
        })

flip_df = pd.DataFrame(flip_by_tier)
print('Counterfactual results by risk tier:')
print(flip_df.round(3).to_string(index=False))

if len(flip_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].bar(flip_df['Tier'], flip_df['Flip Rate'],
                color=['#ef5350', '#ffa726', '#66bb6a'], edgecolor='white')
    axes[0].set_ylabel('Flip Rate')
    axes[0].set_ylim(0, 1)
    axes[0].set_title('Counterfactual Flip Rate by Risk Tier', fontweight='bold')

    axes[1].bar(flip_df['Tier'], flip_df['Mean Δ Prob'],
                color=['#ef5350', '#ffa726', '#66bb6a'], edgecolor='white')
    axes[1].set_ylabel('Mean Δ Probability')
    axes[1].set_title('Mean Probability Gain by Risk Tier', fontweight='bold')

    plt.suptitle('Counterfactual Analysis — Random Forest', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/counterfactual_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Fairness Analysis Across All Demographic Axes

In [ ]:
group_axes = ['gender', 'first_gen', 'ses_index']
group_labels = {'gender': 'Gender', 'first_gen': 'First-Gen Status', 'ses_index': 'SES Tier'}

fairness_summary = []

for group_col in group_axes:
    result = compute_fairness_metrics(rf_pipeline, X_test, y_test, demo_test, group_col)
    flags  = result['fairness_flags']
    fairness_summary.append({
        'Axis':                  group_labels[group_col],
        'Disparate Impact':      result['disparate_impact_ratio'],
        'DI OK (≥0.8)':          '✅' if flags['disparate_impact_ok']   else '❌',
        'EO Gap':                result['equal_opportunity_gap'],
        'EO OK (≤0.10)':         '✅' if flags['equal_opportunity_ok']  else '❌',
        'DP Gap':                result['demographic_parity_gap'],
        'DP OK (≤0.10)':         '✅' if flags['demographic_parity_ok'] else '❌',
    })

print('Fairness Summary — Random Forest')
print(pd.DataFrame(fairness_summary).round(4).to_string(index=False))

---
**Notebook complete.** Proceed to `04_case_studies.ipynb` for end-to-end student case studies.